# DICES Client Reference

This notebook is a reference for the `dicesapi` Python package: how to connect
to the database, search for records, and work with the objects you get back.

Unlike the worked-example notebooks (`Ex_01`, `Ex_02`, etc.), this one is meant
to be looked things up in rather than read start to finish. Search parameters
for each endpoint are queried live from the server, so the lists you see here
always match what the database actually supports -- they won't go stale the
way a hand-written list would.


## Installation

**If you're using Google Colab**, run the cell below to install `dices-client`.
If you're running locally and already have it installed (e.g. in a venv),
you can skip this.

In [ ]:
!pip install git+https://github.com/cwf2/dices-client

## Connecting to the database

Everything starts with a `DicesAPI` object, which represents your connection
to the server.

##### Arguments

- `dices_api`: base URL of the API. Default is `'http://db.dices.mta.ca/api/'`
- `logfile`: optional path to a local file for log messages. If omitted, log
  messages go to STDERR (and by default aren't shown at all -- see
  [the `logging` docs](https://docs.python.org/3/library/logging.html) if you
  want to see them).
- `progress_class`: optional progress-bar class, used by methods that
  retrieve a lot of paged data.

In [ ]:
from dicesapi import DicesAPI

api = DicesAPI()

## Searching the database

Each of the methods below (`getAuthors()`, `getWorks()`, etc.) retrieves
records from one endpoint of the API. They all accept the same kind of
keyword arguments: one per searchable field. For example:

```python
api.getAuthors(name='Homer')
```

The full set of search fields differs by endpoint, and can change over time
as the database adds new metadata. Rather than maintain a separate
hand-written list here (which is exactly what got out of date before), you
can ask the server directly with `api.printSearchFields(endpoint)`.

Let's start with `'authors'`, since it has the fewest fields and is the
easiest to read.

In [ ]:
api.printSearchFields('authors')

Each line is one search parameter you can pass as a keyword argument to
`api.getAuthors()`. The word in parentheses is the parameter's *type*:

- `(string)` -- text. Most string fields do substring (partial) matching, so
  `name='Hom'` will match "Homer".
- `(integer)` -- a whole number, usually an internal database ID.
- `(boolean)` -- `True` or `False`.

Some fields are *choice* fields, restricted to a fixed set of values. For
those, `printSearchFields()` also lists the allowed values and what they
mean, as `value=Label`. You'll see this below with `works`, which has a
`lang` field.

So, reading the output above: you can search authors by `name` (partial
match), `id`, `wd` (WikiData ID), or `urn` (CTS URN), as well as by
properties of works attributed to them (`work_id`, `work_title`, etc.).

In [ ]:
# get all authors named (or containing) 'Hom'
authors = api.getAuthors(name='Hom')
authors

## Works

A `Work` is a single poem. Let's check its search fields -- this is a good
example of a *choice* field (`lang`).

In [ ]:
api.printSearchFields('works')

In [ ]:
# all Latin epics
works = api.getWorks(lang='latin')
works

## Characters and Character Instances

DICES distinguishes between a **Character** -- the underlying mythological
identity, like Athena -- and a **Character Instance**: a specific appearance
of that character in a specific context. The same `Character` can have
several `CharacterInstance`s, including across different works, or even
within a single one. For example, Athena disguised as Mentor in the
*Odyssey* is a different instance from Athena appearing in her own form.

Both endpoints have more search fields than `authors`/`works`, since they
also let you filter by properties of the work and author a character
appears in. Don't worry about memorizing them -- that's what
`printSearchFields()` is for.

In [ ]:
api.printSearchFields('characters')

In [ ]:
# mortal women who speak in the Odyssey
characters = api.getCharacters(work_title='Odyssey', being='mortal', gender='female')
characters

In [ ]:
api.printSearchFields('instances')

In [ ]:
# all instances of Ares
instances = api.getInstances(char_name='Ares')
instances

## Speeches and Speech Clusters

A **Speech** is a single, continuous turn of direct discourse. A **Speech
Cluster** groups together the speeches that make up one conversation or
exchange.

These two endpoints have the most search fields of any in the database,
since a speech can be filtered by properties of its speaker(s), addressee(s),
work, author, *and* the speech itself. The output of `printSearchFields()`
is correspondingly long -- that's expected, not a sign you're doing
something wrong. A few tips for navigating it:

- Fields prefixed `spkr_` and `addr_` mirror each other -- e.g. `spkr_gender`
  and `addr_gender` work the same way, just for speaker vs. addressee.
- Fields with `_inst_` in the name (e.g. `spkr_inst_name`) search the
  character *instance* -- the specific appearance in this speech. Fields
  without it (e.g. `spkr_name`) search the underlying *character* instead.
  These usually give the same results, but can differ for disguises or
  name changes.
- `work_*` and `author_*` fields let you filter by the speech's containing
  work/author without a separate query.

In [ ]:
api.printSearchFields('speeches')

In [ ]:
# all speeches in the Iliad where Aphrodite is addressed
speeches = api.getSpeeches(addr_name='Aphrodite', work_title='Iliad')
speeches

In [ ]:
api.printSearchFields('clusters')

In [ ]:
# all conversations in which Ascanius speaks
clusters = api.getClusters(spkr_name='Ascanius')
clusters

## Working with results: the record classes

Each `get*()` method above returns a *group* object (an `AuthorGroup`,
`WorkGroup`, etc.) holding individual records (`Author`, `Work`, etc.).
The sections below describe the properties available on each record type,
and the convenience methods available on each group type.

Every record has both an `id` (an internal numeric identifier) and a
`public_id` (a short, stable hex code, e.g. `'0A63'`). `public_id` is the
one to use when citing a record or looking one up later -- it's what
appears in the database's citable URNs, and it's what each object's
`repr()` shows. `id` is kept around for backwards compatibility with
existing code, but new code should prefer `public_id`.

Note: the properties listed here are the Python attributes available on
each object -- which aren't always named identically to the search
parameters above (e.g. `addr_name` is a search parameter, but the
corresponding attribute on a `Speech` is `speech.addr`, a list of
`CharacterInstance` objects). A handful of fields visible in the raw API
response aren't yet mapped onto convenient Python attributes (e.g. `tags`,
`notes`) -- if you need one of these, you can find it in the record's
`_attributes` dict.

### Author

#### Properties

- `public_id`: a short, stable identifier for the author -- prefer this
  for citation and lookup
- `id`: an internal numeric identifier, kept for backwards compatibility
- `name`: the author's name
- `wd`: a WikiData ID for the author, if we have it
- `urn`: a CTS URN for the author, if we have it

#### Example

In [ ]:
# get some author records
authors = api.getAuthors(work_title='Argonautica')
authors

### Work

#### Properties

- `public_id`: a short, stable identifier for the work -- prefer this for
  citation and lookup
- `id`: an internal numeric identifier, kept for backwards compatibility
- `title`: the work's title
- `wd`: a WikiData ID for the work, if we have it
- `urn`: a CTS URN for the work, if we have it
- `author`: the associated `Author` object
- `lang`: the work's language, one of `'greek'` or `'latin'`

#### Example

In [ ]:
works = api.getWorks(lang='latin')
works

### Character

#### Properties

- `public_id`: a short, stable identifier for the character -- prefer this
  for citation and lookup
- `id`: an internal numeric identifier, kept for backwards compatibility
- `name`: the character's name
- `being`: one of `'divine'`, `'mortal'`, `'creature'`, `'other'` [1]
- `number`: one of `'individual'`, `'collective'`
- `gender`: one of `'male'`, `'female'`, `'x'`, `'none'` [2]
- `wd`: a WikiData ID for the character, if we have one
- `manto`: a MANTO ID for the character, if we have one
- `tt`: a ToposText ID for the character, if we have one

#### Notes

1. While humans, monsters, and the Olympian gods are usually straightforward
   to classify, miscellaneous nymphs and offspring of minor deities can be
   ambiguous. If you feel a character is misclassified, or find an
   inconsistency in the scheme, please let us know.
2. `'x'` is used for mixed-gender collectives and characters classed as
   non-binary; `'none'` is for characters where gender doesn't apply,
   generally inanimate objects. If you have alternative schemes that might
   be more useful, we'd like to hear about them.

#### Example

In [ ]:
# mortal women who speak in the Odyssey
characters = api.getCharacters(work_title='Odyssey', being='mortal', gender='female')
characters

### Character Instance

#### Properties

- `public_id`: a short, stable identifier for the instance -- prefer this
  for citation and lookup
- `id`: an internal numeric identifier, kept for backwards compatibility
- `context`: a description of the context in which the instance occurs
  (defaults to the work title)
- `name`: the name under which the character appears in this context [1]
- `char`: the underlying `Character`, or `None` [2]
- `being`, `number`, `gender`: as for `Character`, but specific to this
  instance [1]
- `disg` / `anon`: the instance's disguise description and anonymous-speaker
  flag
- `wd`, `manto`, `tt`: pass through to the underlying `Character` [3]

#### Notes

1. An instance's `name`, `being`, `number`, and `gender` can differ from
   those of its underlying character -- e.g. an instance might be named
   "Mentor" while its underlying character is "Athena".
2. Some instances have no underlying character at all -- this represents an
   anonymous speaker or addressee (`anon=True`), such as an unnamed soldier.
3. These will be `None` if there's no underlying character, or if the
   character lacks the attribute.

#### Example

In [ ]:
# all instances of Ares
instances = api.getInstances(char_name='Ares')
instances

### Speech

#### Properties

- `public_id`: a short, stable identifier for the speech -- prefer this
  for citation and lookup
- `id`: an internal numeric identifier, kept for backwards compatibility
- `cluster`: the `SpeechCluster` this speech belongs to
- `seq`: an integer for ordering all speeches within a work
- `l_fi`, `l_la`: the loci of the speech's first and last lines, as strings
- `l_range`: `l_fi` and `l_la` joined with `'-'`
- `spkr`, `addr`: lists of `CharacterInstance` objects for the speaker(s)
  and addressee(s)
- `part`: this speech's turn number within its cluster
- `level`: nesting level (`0` = main narrative, `1`+ = a speech embedded
  within another speech)
- `type`: one of `'S'` (soliloquy), `'M'` (monologue), `'D'` (dialogue),
  `'G'` (general)
- `work`: the associated `Work`
- `author`, `lang`: shortcuts through to `work.author`, `work.lang`
- `urn`: CTS URN for the passage (built from `work.urn` and `l_range`)

#### Methods

- `getSpkrString(sep=', ')` / `getAddrString(sep=', ')`: speaker/addressee
  names joined into a single string
- `splitLocus()`, `getLineNo()`, `getPrefix()`, `isMultiPrefix()`: helpers
  for parsing `l_fi`/`l_la` when they include a prefix (e.g. book number)
  before the line number
- `isRepliedTo()`, `isInterrupted()`, `isInterruption()`: check this
  speech's relationship to others in its cluster (each makes an API call)
- `fetchPassage()`: download the speech's text from a CTS server. Requires
  `api.initializeCts()` to have been called first -- see the text-retrieval
  notebook (`Ex_03 - Text.ipynb`) for details. This replaces the older,
  now-removed `getCTS()` method.

#### Example

You can look up a specific speech later by its `public_id`, using
`filterBy()` on a `SpeechGroup`:

```python
speeches.filterBy('public_id', 'E131')[0]
# <Speech E131: Achilleid 1.339-1.342>
```

In [ ]:
# all speeches in the Iliad where Aphrodite is addressed
speeches = api.getSpeeches(addr_name='Aphrodite', work_title='Iliad')
speeches

### Speech Cluster

#### Properties

- `public_id`: a short, stable identifier for the cluster -- prefer this
  for citation and lookup
- `id`: an internal numeric identifier, kept for backwards compatibility

#### Methods

- `getSpeeches()`: all speeches in this cluster, as a `SpeechGroup`
- `getFirstSpeech()`: just the first speech, as a `Speech`
- `countReplies()`, `countInterruptions()`: counts of replies/interruptions
  within the cluster

#### Notes

1. All of the methods above make an API call in the background.

#### Example

In [ ]:
# all conversations in which Ascanius speaks
clusters = api.getClusters(spkr_name='Ascanius')
clusters

## Group classes

Each `get*()` method returns a *group* -- a list-like container with extra
convenience methods. Two flavors recur throughout:

- `get<X>s()`: pull the `<x>` attribute off every member, as a plain list
- `filter<X>s(values)`: return a new group containing only members whose
  `<x>` attribute is in `values`

Some `get<X>s()` methods accept `flatten=True`, which deduplicates the
result and returns it as a group of its own (e.g. `WorkGroup.getAuthors()`
can return either a list of `Author` objects with duplicates, or a
deduplicated `AuthorGroup`).

### AuthorGroup

- `getIDs()`, `getPublicIds()`, `getNames()`, `getWDs()`, `getUrns()`
- `filterIDs(list)`, `filterPublicIds(list)`, `filterNames(list)`,
  `filterWDs(list)`, `filterUrns(list)`

### WorkGroup

- `getIDs()`, `getPublicIds()`, `getTitles()`, `getWDs()`, `getUrns()`,
  `getLangs()`
- `getAuthors(flatten=False)`
- `filterIDs(list)`, `filterPublicIds(list)`, `filterTitles(list)`,
  `filterWDs(list)`, `filterUrns(list)`, `filterAuthors(list)`,
  `filterLangs(list)`

### CharacterGroup

- `getIDs()`, `getPublicIds()`, `getNames()`, `getBeings()`,
  `getNumbers()`, `getGenders()`, `getWDs()`, `getMantos()`
- `filterIDs(list)`, `filterPublicIds(list)`, `filterNames(list)`,
  `filterBeings(list)`, `filterNumbers(list)`, `filterGenders(list)`,
  `filterWDs(list)`, `filterMantos(list)`

### CharacterInstanceGroup

- `getIDs()`, `getPublicIds()`, `getNames()`, `getContexts()`,
  `getBeings()`, `getNumbers()`, `getGenders()`, `getDisgs()`, `getAnons()`
- `getChars(flatten=False)`
- `filterIDs(list)`, `filterPublicIds(list)`, `filterNames(list)`,
  `filterContexts(list)`, `filterBeings(list)`, `filterNumbers(list)`,
  `filterGenders(list)`, `filterChars(list)`

### SpeechGroup

- `getIDs()`, `getPublicIds()`, `getSeqs()`, `getL_fis()`, `getL_las()`,
  `getParts()`, `getTypes()`
- `getClusters(flatten=False)`, `getWorks(flatten=False)`,
  `getSpkrs(flatten=False)`, `getAddrs(flatten=False)`
- `filterIDs(list)`, `filterPublicIds(list)`, `filterClusters(list)`,
  `filterSeqs(list)`, `filterL_fis(list)`, `filterL_ls(list)`,
  `filterParts(list)`, `filterTypes(list)`, `filterWorks(list)`,
  `filterSpkrs(list)`, `filterAddrs(list)`, `filterSpkrInstances(list)`,
  `filterAddrInstances(list)`

`filterSpkrs()`/`filterAddrs()` match on the underlying `Character`;
`filterSpkrInstances()`/`filterAddrInstances()` match on the specific
`CharacterInstance`.

### SpeechClusterGroup

- `getIDs()`, `getPublicIds()`
- `filterIDs(list)`, `filterPublicIds(list)`

## Beyond search: text and NLP

Two optional pieces of `dicesapi` aren't covered here in detail, since they
have their own dedicated notebooks:

- **Text retrieval** (`dicesapi.text`): call `api.initializeCts()` once, then
  `speech.fetchPassage()` to download a speech's text from a CTS server.
  See `Ex_03 - Text.ipynb`.
- **NLP** (`dicesapi.nlp_spacy`): call `api.initializeNlp(latin_model=...,
  greek_model=...)` to load spaCy models, then run them on a passage. See
  `NLP.ipynb`.

There are also optional modules for linking DICES characters to external
databases: `dicesapi.wikidata` (requires `pip install dices-client[wikidata]`)
and `dicesapi.manto`, covered in `Ex_04 - MANTO.ipynb` and
`Ex_05 - More with MANTO.ipynb`.